# Does a direct sum reproduce the production bin?

The Goertzel kernel and the NumPy kernel are written to compute the same Fourier coefficient, up to a phase that drops out of the power. This notebook checks both against a direct sum that calls neither kernel.

For one segment the direct coefficient is

$$X = \sum_n y[n]\,e^{-i\omega n}$$

after the detrend and the window. `order=-1` leaves the samples as they are. `order=0` subtracts the mean. `order=1` or `order=2` subtracts the projection onto polynomials in the time coordinate $t \in [-1, 1]$. The production Goertzel value equals this sum times $e^{i\omega(L-1)}$. That phase is common to both channels, so it cancels in $|X|^2$ and in $X Y^{*}$. The comparison below is on `XX`, the mean of $|X|^2$.

The tolerance written next to the test in `tests/unit/test_direct_dft.py` is $8 L \varepsilon \sum |y_{\mathrm{windowed}}|$ on the coefficient. $\varepsilon$ is machine precision. This notebook prints the relative error of the averaged power, which should sit at roundoff. The cases are a short segment and a long one, an integer bin, a fractional bin, and a frequency below one bin. The window is Hann. This notebook compares powers. The one-sided factor of 2 appears only when a density is formed, and no density is formed here.


In [ ]:
import numpy as np
from speckit import compute_single_bin


def direct_coefficient(segment, window, omega, order):
    """Direct sum after the documented detrend. Does not call Goertzel."""
    y = np.array(segment, dtype=np.float64, copy=True)
    if order == 0:
        y = y - y.mean()
    elif order in (1, 2):
        t_seg = np.linspace(-1.0, 1.0, y.size)
        if order == 1:
            basis = np.column_stack([np.ones(y.size), t_seg])
        else:
            basis = np.column_stack([np.ones(y.size), t_seg, t_seg * t_seg])
        q, _ = np.linalg.qr(basis, mode="reduced")
        y = y - q @ (q.T @ y)
    n = np.arange(y.size, dtype=np.float64)
    return np.dot(y * window, np.exp(-1j * omega * n))


def relative_power(length, cycles, order, seed):
    hop = length // 2
    n = length + hop
    fs = float(length)  # one bin is 1 Hz, so cycles is the frequency in Hz
    rng = np.random.default_rng(seed)
    t = np.arange(n)
    x = 0.1 * rng.standard_normal(n) + np.cos(2.0 * np.pi * 0.13 * t)
    freq = cycles * fs / length
    window = np.hanning(length)
    omega = 2.0 * np.pi * freq / fs
    single = compute_single_bin(
        x, fs, freq=freq, L=length, order=order, win="hann", olap=0.5
    )
    starts = np.asarray(single.D[0], dtype=np.int64)
    coefficients = [
        direct_coefficient(x[int(s) : int(s) + length], window, omega, order)
        for s in starts
    ]
    direct_power = np.mean(np.abs(coefficients) ** 2)
    return abs(float(single.XX[0]) - direct_power) / direct_power


cases = (
    ("short integer", 32, 4.0),
    ("short fractional", 32, 4.5),
    ("short near zero", 32, 0.15),
    ("long integer", 2048, 128.0),
    ("long fractional", 2048, 128.5),
    ("long near zero", 2048, 0.2),
)
for name, length, cycles in cases:
    for order in (-1, 0, 1, 2):
        relative = relative_power(length, cycles, order, seed=20261025 + length)
        print(f"{name:18} order {order:2d}  relative power error {relative:.3e}")


## Reading the result

Each line is one detrend order, one segment length, and one target bin. The printed number is the relative difference between production `XX` and the mean of $|X|^2$ from the direct sum. On the short segments it is about $10^{-16}$ to $10^{-14}$. On the long segment, at a fraction of a bin, it reaches a few times $10^{-11}$. That gap is the recurrence against a plain sum. A result near $10^{-1}$ would mean the production bin is a different coefficient. Both sides of this comparison are powers, so the Goertzel phase factor is absent from both.

`fs` is set equal to the segment length, so an integer `cycles` value is an integer DFT bin, at 1 Hz per bin. The near-zero rows use a fraction of a bin. The notebook prints powers, and it forms no one-sided density.
